# 01 Generate Raw Source Tables

This notebook explains how the raw source tables for the AgriCredit Resilience project were created.

The project does not use real household loan default records because such data is difficult to access publicly and may contain sensitive personal information. Instead, this project uses semi-synthetic raw-like data.

The raw tables are generated using information and patterns inspired by Myanmar-focused reports on rural livelihoods, agriculture, food security, market access, humanitarian assistance, and household vulnerability.

The reusable Python script used to generate the raw tables is:

`src/generate_raw_data.py`

This notebook does not contain the full raw data generation logic. The main generation logic is kept in the script so the process can be reproduced easily.

## Notebook and Script Workflow

Unlike the later notebooks, this notebook does not build the raw data generation logic step by step. The raw data generation process is stored in `src/generate_raw_data.py` because it is a reusable utility script.

This notebook focuses on explaining why semi-synthetic raw data was created, confirming which raw files were generated, and inspecting the raw outputs before data understanding and preparation.

## Why Semi-Synthetic Raw Data Was Used

Real financial vulnerability and loan repayment data is usually not publicly available. It can also include sensitive household information.

To still build a realistic machine learning project, this project creates semi-synthetic raw source tables. The data is synthetic, but the variables are informed by real development and humanitarian themes in Myanmar, such as:

- household income and debt
- market access
- food coping strategies
- agriculture and crop damage
- minimum expenditure basket values
- assistance coverage gaps
- price changes and market pressure

The purpose is not to claim that this is real survey data. The purpose is to build a responsible, explainable, and reproducible ML workflow for a realistic development problem.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
SCRIPT_PATH = PROJECT_ROOT / "src" / "generate_raw_data.py"

RAW_DATA_DIR, SCRIPT_PATH

(PosixPath('/Users/isaacaung/Desktop/agricredit-resilience/data/raw'),
 PosixPath('/Users/isaacaung/Desktop/agricredit-resilience/src/generate_raw_data.py'))

## Reproducible Raw Data Generation

The raw data can be regenerated from the project root using this command:

```bash
python src/generate_raw_data.py

In [3]:
RAW_DATA_DIR.exists()

True

In [4]:
raw_files = sorted(RAW_DATA_DIR.glob("*.csv"))

raw_file_names = [file.name for file in raw_files]
raw_file_names

['agriculture_raw.csv',
 'assistance_coverage_raw.csv',
 'coping_raw.csv',
 'households_raw.csv',
 'market_prices_raw.csv',
 'meb_values_raw.csv']

In [5]:
raw_table_summary = []

for file in raw_files:
    df = pd.read_csv(file)
    raw_table_summary.append({
        "file_name": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

raw_table_summary_df = pd.DataFrame(raw_table_summary)
raw_table_summary_df

,file_name,rows,columns
0,agriculture_raw.csv,1500,7
1,assistance_coverage_raw.csv,14,6
2,coping_raw.csv,1500,7
3,households_raw.csv,1500,13
4,market_prices_raw.csv,14,6
5,meb_values_raw.csv,14,3


## Raw Source Tables

The project uses six raw source tables:

1. `households_raw.csv`  
   Household-level financial and demographic information.

2. `agriculture_raw.csv`  
   Farming status, crop type, farm size, irrigation access, fertilizer cost, and crop damage.

3. `coping_raw.csv`  
   Food coping strategy indicators and basic needs information.

4. `market_prices_raw.csv`  
   State/region-level price changes and market pressure indicators.

5. `meb_values_raw.csv`  
   Minimum Expenditure Basket and cash assistance reference values.

6. `assistance_coverage_raw.csv`  
   Humanitarian/livelihood assistance coverage and response gap indicators.

In [6]:
households = pd.read_csv(RAW_DATA_DIR / "households_raw.csv")
agriculture = pd.read_csv(RAW_DATA_DIR / "agriculture_raw.csv")
coping = pd.read_csv(RAW_DATA_DIR / "coping_raw.csv")
market_prices = pd.read_csv(RAW_DATA_DIR / "market_prices_raw.csv")
meb_values = pd.read_csv(RAW_DATA_DIR / "meb_values_raw.csv")
assistance_coverage = pd.read_csv(RAW_DATA_DIR / "assistance_coverage_raw.csv")

In [7]:
households.head()

,household_id,state_region,township,household_size,monthly_income,has_debt,total_debt,monthly_debt_repayment,savings_duration_weeks,market_access,female_headed_household,disability_present,displacement_status
0,HH00001,Magway,Magway,6,610149.0,yes,851624,150329,1,Yes,No,No,IDP
1,HH00002,Ayeyarwady,Pathein,8,284061.0,Yes,801819,95454,4,No,No,No,Resident
2,HH00003,Shan North,Lashio,4,50000.0,No,0,0,2,Unknown,No,No,Resident
3,HH00004,Kayin,Hpa-An,4,101947.0,Yes,667989,60245,2,Yes,No,No,Resident
4,HH00005,Kayin,Myawaddy,6,675893.0,Yes,593253,81094,1,N,No,No,Returnee


In [8]:
agriculture.head()

,household_id,is_farming_household,main_crop,farm_size_acres,irrigation_access,fertilizer_cost,crop_damage_recent
0,HH00001,Yes,Rice,3.80,No,88356.0,No
1,HH00002,Yes,Rice,2.90,No,236635.0,No
2,HH00003,Yes,Rice,0.63,No,185682.0,Yes
3,HH00004,No,NaN,0.00,No,0.0,No
4,HH00005,No,NaN,0.00,No,0.0,No


In [9]:
coping.head()

,household_id,less_preferred_food_days,borrowed_food_days,reduced_meals_days,reduced_portion_days,adults_reduced_for_children_days,basic_needs_met
0,HH00001,0.0,0.0,0.0,2.0,0.0,Yes mostly
1,HH00002,6.0,0.0,2.0,6.0,1.0,Yes mostly
2,HH00003,5.0,2.0,1.0,3.0,3.0,Only some
3,HH00004,1.0,1.0,2.0,1.0,3.0,Yes mostly
4,HH00005,7.0,NaN,5.0,3.0,1.0,Only some


In [10]:
market_prices.head()

,state_region,month,basic_food_basket_change_1y,rice_price_change_1y,fuel_price_change_1y,market_disruption_level
0,Sagaing,2025-04,34.4,10.9,34.3,Medium
1,Magway,2025-04,61.4,11.4,8.6,Medium
2,Mandalay,2025-04,11.3,9.2,35.3,Low
3,Chin,2025-04,39.8,12.5,25.0,Medium
4,Kachin,2025-04,15.2,18.4,1.1,Low


In [11]:
meb_values.head()

,state_region,mpca_per_person,cash_food_assistance_per_person
0,Bago,55000,35000
1,Chin,95000,65000
2,Kachin,65000,50000
3,Kayah,80000,55000
4,Kayin,65000,40000


In [12]:
assistance_coverage.head()

,state_region,people_targeted,people_reached,coverage_rate,response_gap,number_of_partners_active
0,Sagaing,178672,122516,0.686,56156,12
1,Magway,24770,9411,0.380,15359,21
2,Mandalay,147038,87058,0.592,59980,26
3,Chin,104011,39085,0.376,64926,10
4,Kachin,59479,21163,0.356,38316,20


## Raw Data Messiness

The raw tables intentionally include some realistic data quality issues. This is important because a real machine learning project usually does not start with perfectly clean data.

Examples of intentional messiness include:

- inconsistent yes/no values such as `Yes`, `Y`, `No`, and `N`
- inconsistent state/region naming
- missing values
- money values that may need cleaning
- separate tables that need to be merged later

These issues allow the next notebooks to demonstrate proper data understanding and data preparation.

In [13]:
messiness_checks = {
    "state_region_values": households["state_region"].unique()[:10],
    "has_debt_values": households["has_debt"].unique(),
    "market_access_values": households["market_access"].unique(),
    "main_crop_missing_count": agriculture["main_crop"].isna().sum(),
    "monthly_income_missing_count": households["monthly_income"].isna().sum(),
}

messiness_checks

{'state_region_values': <ArrowStringArray>
 [     'Magway',  'Ayeyarwady',  'Shan North',       'Kayin',         'Mon',
     'Mandalay',     'Rakhine', 'Tanintharyi',        'Bago',       'Kayah']
 Length: 10, dtype: str,
 'has_debt_values': <ArrowStringArray>
 ['yes', 'Yes', 'No', 'no', 'N', 'Y']
 Length: 6, dtype: str,
 'market_access_values': <ArrowStringArray>
 ['Yes', 'No', 'Unknown', 'N', 'Y']
 Length: 5, dtype: str,
 'main_crop_missing_count': np.int64(358),
 'monthly_income_missing_count': np.int64(60)}

## Relationship Between Notebook and Script

This notebook is used for explanation and portfolio documentation.

The reusable script is used for automation:

`src/generate_raw_data.py`

This means the project has both:

- a notebook that explains what was generated and why
- a script that can reproduce the raw data generation process

This keeps the project clear, professional, and reproducible.

## Summary

In this notebook, the raw source tables were inspected and explained.

The generated raw data provides the foundation for the next stage:

`02_data_understanding.ipynb`

In the next notebook, the raw tables are explored more deeply to identify missing values, inconsistent categories, distributions, and preparation needs before building the final analytical base table.